# **CLEAN BERITA TF IDF DAN WORD EMBEDDING**

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, sys, time

In [2]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download resource NLTK
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Load File

In [3]:
# Load file hasil crawling
df = pd.read_csv("/content/tempo_berita.csv")

# Gabungkan kolom judul dan isi berita
df["text"] = df["judul_berita"].astype(str) + " " + df["isi_berita"].astype(str)

# Cek hasil gabungan
print("Jumlah data:", len(df))
df[["judul_berita", "isi_berita", "text"]]

Jumlah data: 900


,judul_berita,isi_berita,text
0,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,TENTARA Nasional Indonesia atau TNI menggelar ...,Ketika Para Jenderal Ikut Defile di HUT ke-80 ...
1,Prabowo Minta Semua Pesantren Didata setelah P...,PRESIDENPrabowoSubianto memerintahkan semua po...,Prabowo Minta Semua Pesantren Didata setelah P...
2,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,PRESIDEN Prabowo Subianto memerintahkan Pangli...,Prabowo: Utamakan Kompetensi Prajurit Dibandin...
3,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,KEMENTERIAN Komunikasi dan Digital (Kemenkomdi...,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...
4,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,MANTAN Presiden Megawati Soekarnoputri dan Jok...,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...
...,...,...,...
895,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ADA dua kondisi yang kini melekat padaHarry Ka...,Ketika Harry Kane Memecahkan Rekor Gol Cristia...
896,Peluang Timnas Indonesia Lewati Hadangan Arab ...,PENGAMAT sepak bola Tanah Air Kesit Budi Hando...,Peluang Timnas Indonesia Lewati Hadangan Arab ...
897,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,"DALAM usia 40 tahun,Cristiano Ronaldomasih mam...",Seperti Apa Ketajaman Cristiano Ronaldo Bersam...
898,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,"BADAN sepak bola dunia,FIFA, menjatuhkan sanks...",FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...


# PREPROCESSING

## Bersihkan Teks (Cleaning)

In [4]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()  # ubah ke huruf kecil
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)  # hapus URL
    text = re.sub(r"\d+", "", text)  # hapus angka
    text = text.translate(str.maketrans("", "", string.punctuation))  # hapus tanda baca
    text = re.sub(r"\s+", " ", text).strip()  # hapus spasi berlebih
    return text

df["clean_text"] = df["text"].apply(clean_text)

# Tampilkan contoh hasil cleaning
df[["text", "clean_text"]]

,text,clean_text
0,Ketika Para Jenderal Ikut Defile di HUT ke-80 ...,ketika para jenderal ikut defile di hut ke tni...
1,Prabowo Minta Semua Pesantren Didata setelah P...,prabowo minta semua pesantren didata setelah p...
2,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,prabowo utamakan kompetensi prajurit dibanding...
3,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,kemenkomdigi permintaan data ke tiktok hanya u...
4,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,megawati dan jokowi tak hadir di hut ke tni di...
...,...,...
895,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ketika harry kane memecahkan rekor gol cristia...
896,Peluang Timnas Indonesia Lewati Hadangan Arab ...,peluang timnas indonesia lewati hadangan arab ...
897,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,seperti apa ketajaman cristiano ronaldo bersam...
898,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,fifa jatuhkan sanksi untuk malaysia dan pemain...


## Tokenisasi

In [5]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
df["tokens"] = df["clean_text"].apply(word_tokenize)

# Contoh hasil tokenisasi
df[["text", "clean_text", "tokens"]]

,text,clean_text,tokens
0,Ketika Para Jenderal Ikut Defile di HUT ke-80 ...,ketika para jenderal ikut defile di hut ke tni...,"[ketika, para, jenderal, ikut, defile, di, hut..."
1,Prabowo Minta Semua Pesantren Didata setelah P...,prabowo minta semua pesantren didata setelah p...,"[prabowo, minta, semua, pesantren, didata, set..."
2,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,prabowo utamakan kompetensi prajurit dibanding...,"[prabowo, utamakan, kompetensi, prajurit, diba..."
3,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,kemenkomdigi permintaan data ke tiktok hanya u...,"[kemenkomdigi, permintaan, data, ke, tiktok, h..."
4,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,megawati dan jokowi tak hadir di hut ke tni di...,"[megawati, dan, jokowi, tak, hadir, di, hut, k..."
...,...,...,...
895,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ketika harry kane memecahkan rekor gol cristia...,"[ketika, harry, kane, memecahkan, rekor, gol, ..."
896,Peluang Timnas Indonesia Lewati Hadangan Arab ...,peluang timnas indonesia lewati hadangan arab ...,"[peluang, timnas, indonesia, lewati, hadangan,..."
897,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,seperti apa ketajaman cristiano ronaldo bersam...,"[seperti, apa, ketajaman, cristiano, ronaldo, ..."
898,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,fifa jatuhkan sanksi untuk malaysia dan pemain...,"[fifa, jatuhkan, sanksi, untuk, malaysia, dan,..."


## STOPWORDS

In [7]:
stop_words = set(stopwords.words("indonesian"))

df["tokens"] = df["tokens"].apply(
    lambda x: [word for word in x if word not in stop_words and len(word) > 2]
)

# Contoh hasil akhir token
df[["text", "clean_text", "tokens"]]

,text,clean_text,tokens
0,Ketika Para Jenderal Ikut Defile di HUT ke-80 ...,ketika para jenderal ikut defile di hut ke tni...,"[jenderal, defile, hut, tni, tentara, nasional..."
1,Prabowo Minta Semua Pesantren Didata setelah P...,prabowo minta semua pesantren didata setelah p...,"[prabowo, pesantren, didata, ponpes, khoziny, ..."
2,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,prabowo utamakan kompetensi prajurit dibanding...,"[prabowo, utamakan, kompetensi, prajurit, diba..."
3,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,kemenkomdigi permintaan data ke tiktok hanya u...,"[kemenkomdigi, permintaan, data, tiktok, penga..."
4,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,megawati dan jokowi tak hadir di hut ke tni di...,"[megawati, jokowi, hadir, hut, tni, monas, man..."
...,...,...,...
895,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ketika harry kane memecahkan rekor gol cristia...,"[harry, kane, memecahkan, rekor, gol, cristian..."
896,Peluang Timnas Indonesia Lewati Hadangan Arab ...,peluang timnas indonesia lewati hadangan arab ...,"[peluang, timnas, indonesia, lewati, hadangan,..."
897,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,seperti apa ketajaman cristiano ronaldo bersam...,"[ketajaman, cristiano, ronaldo, nassr, musim, ..."
898,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,fifa jatuhkan sanksi untuk malaysia dan pemain...,"[fifa, jatuhkan, sanksi, malaysia, pemain, nat..."


## SIMPAN HASIL PREPROCESSING

In [8]:
output_path = "/content/tempo_preprocessed.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("✅ Preprocessing selesai!")
print(f"Hasil disimpan ke: {output_path}")

✅ Preprocessing selesai!
Hasil disimpan ke: /content/tempo_preprocessed.csv


# TF-IDF

## LOAD

In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Load hasil preprocessing
df = pd.read_csv("/content/tempo_preprocessed.csv")

# Cek isi kolom
print(df.columns)
df

Index(['id_berita', 'judul_berita', 'isi_berita', 'kategori_berita', 'text',
       'clean_text', 'tokens'],
      dtype='object')


,id_berita,judul_berita,isi_berita,kategori_berita,text,clean_text,tokens
0,2076486,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,TENTARA Nasional Indonesia atau TNI menggelar ...,politik,Ketika Para Jenderal Ikut Defile di HUT ke-80 ...,ketika para jenderal ikut defile di hut ke tni...,"['jenderal', 'defile', 'hut', 'tni', 'tentara'..."
1,2076480,Prabowo Minta Semua Pesantren Didata setelah P...,PRESIDENPrabowoSubianto memerintahkan semua po...,politik,Prabowo Minta Semua Pesantren Didata setelah P...,prabowo minta semua pesantren didata setelah p...,"['prabowo', 'pesantren', 'didata', 'ponpes', '..."
2,2076479,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,PRESIDEN Prabowo Subianto memerintahkan Pangli...,politik,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,prabowo utamakan kompetensi prajurit dibanding...,"['prabowo', 'utamakan', 'kompetensi', 'prajuri..."
3,2076473,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,KEMENTERIAN Komunikasi dan Digital (Kemenkomdi...,politik,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,kemenkomdigi permintaan data ke tiktok hanya u...,"['kemenkomdigi', 'permintaan', 'data', 'tiktok..."
4,2076468,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,MANTAN Presiden Megawati Soekarnoputri dan Jok...,politik,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,megawati dan jokowi tak hadir di hut ke tni di...,"['megawati', 'jokowi', 'hadir', 'hut', 'tni', ..."
...,...,...,...,...,...,...,...
895,2073886,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ADA dua kondisi yang kini melekat padaHarry Ka...,sepakbola,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ketika harry kane memecahkan rekor gol cristia...,"['harry', 'kane', 'memecahkan', 'rekor', 'gol'..."
896,2073880,Peluang Timnas Indonesia Lewati Hadangan Arab ...,PENGAMAT sepak bola Tanah Air Kesit Budi Hando...,sepakbola,Peluang Timnas Indonesia Lewati Hadangan Arab ...,peluang timnas indonesia lewati hadangan arab ...,"['peluang', 'timnas', 'indonesia', 'lewati', '..."
897,2073852,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,"DALAM usia 40 tahun,Cristiano Ronaldomasih mam...",sepakbola,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,seperti apa ketajaman cristiano ronaldo bersam...,"['ketajaman', 'cristiano', 'ronaldo', 'nassr',..."
898,2073819,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,"BADAN sepak bola dunia,FIFA, menjatuhkan sanks...",sepakbola,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,fifa jatuhkan sanksi untuk malaysia dan pemain...,"['fifa', 'jatuhkan', 'sanksi', 'malaysia', 'pe..."


## Gabung kolom

In [10]:
# Ubah kolom tokens dari string ke list dan gabungkan jadi kalimat
df["tokens_list"] = df["tokens"].apply(eval)
df["tokens_str"] = df["tokens_list"].apply(lambda x: " ".join(x))

# Cek hasilnya
df[["tokens", "tokens_str"]]

,tokens,tokens_str
0,"['jenderal', 'defile', 'hut', 'tni', 'tentara'...",jenderal defile hut tni tentara nasional indon...
1,"['prabowo', 'pesantren', 'didata', 'ponpes', '...",prabowo pesantren didata ponpes khoziny ambruk...
2,"['prabowo', 'utamakan', 'kompetensi', 'prajuri...",prabowo utamakan kompetensi prajurit dibanding...
3,"['kemenkomdigi', 'permintaan', 'data', 'tiktok...",kemenkomdigi permintaan data tiktok pengawasan...
4,"['megawati', 'jokowi', 'hadir', 'hut', 'tni', ...",megawati jokowi hadir hut tni monas mantan pre...
...,...,...
895,"['harry', 'kane', 'memecahkan', 'rekor', 'gol'...",harry kane memecahkan rekor gol cristiano rona...
896,"['peluang', 'timnas', 'indonesia', 'lewati', '...",peluang timnas indonesia lewati hadangan arab ...
897,"['ketajaman', 'cristiano', 'ronaldo', 'nassr',...",ketajaman cristiano ronaldo nassr musim usia t...
898,"['fifa', 'jatuhkan', 'sanksi', 'malaysia', 'pe...",fifa jatuhkan sanksi malaysia pemain naturalis...


## Hasil TF-IDF

In [11]:
# Buat model TF-IDF
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(df["tokens_str"])

print("TF-IDF shape:", X_tfidf.shape)


TF-IDF shape: (900, 23273)


In [12]:
# Konversi hasil TF-IDF ke DataFrame untuk ditampilkan
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())

# Tampilkan DataFrame
display(tfidf_df)

,aalborg,aan,aarhus,aaron,abad,abadi,abadijaya,abai,abaikan,abang,...,zulkifli,zuma,zumba,zurich,zusfa,zverev,zwelivelile,zwiers,zwolle,ávila
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
896,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
897,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
898,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
# Simpan hasil TF-IDF ke file CSV
output_path_tfidf = "/content/hasil_ekstraksi_fitur.csv"
tfidf_df.to_csv(output_path_tfidf, index=False)

print(f"✅ Hasil ekstraksi fitur (TF-IDF) berhasil disimpan ke: {output_path_tfidf}")

✅ Hasil ekstraksi fitur (TF-IDF) berhasil disimpan ke: /content/hasil_ekstraksi_fitur.csv


# WORD EMBEDDING

In [14]:
!pip install gensim

In [16]:
import pandas as pd
from gensim.models import Word2Vec

# Load data hasil preprocessing
df = pd.read_csv("/content/tempo_preprocessed.csv")

# Ubah kolom tokens jadi list kembali
df["tokens_list"] = df["tokens"].apply(eval)

# Cek hasil
df[["judul_berita", "tokens_list"]]

,judul_berita,tokens_list
0,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,"[jenderal, defile, hut, tni, tentara, nasional..."
1,Prabowo Minta Semua Pesantren Didata setelah P...,"[prabowo, pesantren, didata, ponpes, khoziny, ..."
2,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,"[prabowo, utamakan, kompetensi, prajurit, diba..."
3,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,"[kemenkomdigi, permintaan, data, tiktok, penga..."
4,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,"[megawati, jokowi, hadir, hut, tni, monas, man..."
...,...,...
895,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,"[harry, kane, memecahkan, rekor, gol, cristian..."
896,Peluang Timnas Indonesia Lewati Hadangan Arab ...,"[peluang, timnas, indonesia, lewati, hadangan,..."
897,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,"[ketajaman, cristiano, ronaldo, nassr, musim, ..."
898,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,"[fifa, jatuhkan, sanksi, malaysia, pemain, nat..."


In [17]:
# Dataset untuk pelatihan Word2Vec (list of list of tokens)
sentences = df["tokens_list"].tolist()

print("Jumlah dokumen:", len(sentences))
print("Contoh 1 dokumen:", sentences[0][:20])  # tampilkan 20 kata pertama


Jumlah dokumen: 900
Contoh 1 dokumen: ['jenderal', 'defile', 'hut', 'tni', 'tentara', 'nasional', 'indonesia', 'tni', 'menggelar', 'defile', 'militer', 'peringatan', 'ulang', 'tni', 'ahad', 'oktober', 'jenderal', 'bintang', 'matra', 'defile']


## LATIH MODEL

In [18]:
# Buat model Word2Vec
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,   # dimensi vektor
    window=5,          # konteks kiri-kanan
    min_count=2,       # kata muncul minimal 2x
    sg=1,              # gunakan skip-gram (lebih baik untuk makna kata)
    workers=4,         # gunakan 4 core CPU
    epochs=20          # iterasi pelatihan
)

print("✅ Model Word2Vec berhasil dilatih!")


✅ Model Word2Vec berhasil dilatih!


## CEK HASIL EMBEDDING

In [19]:
# Cek kata mirip
w2v_model.wv.most_similar("presiden", topn=10)

[('subianto', 0.6680493950843811),
 ('wapres', 0.6267814636230469),
 ('prabowo', 0.6247132420539856),
 ('donald', 0.6161702275276184),
 ('assisi', 0.5703600645065308),
 ('lula', 0.5679237842559814),
 ('susilo', 0.5656517744064331),
 ('fattah', 0.5649448037147522),
 ('donaldtrumpuntuk', 0.5574044585227966),
 ('inspektur', 0.5525136590003967)]

In [20]:
# Simpan model
w2v_model.save("/content/word2vec_tempo.model")
print("💾 Model Word2Vec disimpan di: /content/word2vec_tempo.model")


💾 Model Word2Vec disimpan di: /content/word2vec_tempo.model


## EKSTRAK RATA RATA VEKTOR PER DOKUMEN

In [26]:
import numpy as np

def document_vector(tokens):
    """Hitung rata-rata vektor untuk 1 dokumen."""
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if len(vectors) > 0 else np.zeros(w2v_model.vector_size)

# Buat embedding untuk semua dokumen
doc_vectors = np.array([document_vector(tokens) for tokens in df["tokens_list"]])

print("Shape dokumen embedding:", doc_vectors.shape)


Shape dokumen embedding: (900, 100)


In [27]:
# Ubah ke DataFrame
embedding_df = pd.DataFrame(doc_vectors)
embedding_df.insert(0, "doc_id", df["id_berita"])

# Simpan ke CSV
output_path = "/content/word_embedding_result.csv"
embedding_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("✅ Hasil Word Embedding disimpan ke:", output_path)


✅ Hasil Word Embedding disimpan ke: /content/word_embedding_result.csv
